# Phase 2c — PDF Processing Benchmark

Interactive walkthrough of the head-to-head PDF extraction benchmark.
Compares pypdfium2, PyMuPDF, and pypdf on the same PDF, with timing
and output size.

Per the multi-stage plan (see AGENTS.md). Phase 2c.

In [ ]:
# 1. Pick a PDF to benchmark.
import pathlib, sqlite3

PDF_RAW_ROOT = pathlib.Path("data/bi_ep/syllabi_raw")
DB_PATH = pathlib.Path("data/gemini_hackathon.duckdb")

if PDF_RAW_ROOT.exists():
    pdfs = sorted(PDF_RAW_ROOT.rglob("*.pdf"))
    print(f"Available PDFs: {len(pdfs)}")
    for p in pdfs[:5]:
        print(f"  {p.relative_to(PDF_RAW_ROOT.parent.parent):80s} {p.stat().st_size / 1024:8.1f} KiB")
else:
    print("No downloaded PDFs yet. Run `python -m dlt_pipelines.pdf_downloader` first.")
    print("Falling back to a synthesized minimal PDF for the demo.")
    import sys; sys.path.insert(0, 'tests')
    from test_pdf_downloader import _minimal_pdf_bytes
    demo_pdf = pathlib.Path("/tmp/phase2c-demo.pdf")
    demo_pdf.write_bytes(_minimal_pdf_bytes(page_count=5))
    pdfs = [demo_pdf]
    print(f"Demo PDF: {demo_pdf} ({demo_pdf.stat().st_size} bytes)")

In [ ]:
# 2. Run the benchmark on the first available PDF.
from cocoindex_flows.pdf.benchmark import benchmark_pdf
import json

target = pdfs[0]
print(f"Target: {target}\n")
result = benchmark_pdf(target)
print(json.dumps(result, indent=2))

In [ ]:
# 3. Convert a PDF via the canonical pipeline (Phase 2b App).
from cocoindex_flows.pdf.pdf_to_markdown_app import run

stats = run()
print(stats)

## Summary

- **Backend choice**: pypdfium2 is the canonical Phase 2b default (Apache-2.0, fastest text-only, no new deps).
- **CocoIndex app**: when cocoindex IS installed, `cocoindex update pdf_to_markdown_app` runs end-to-end. When missing, the plain `run()` function is the canonical entry point.
- **Output**: `data/bi_ep/syllabi_md/<source_key>/<subject>/<lang>/<sha>.md` — one file per PDF.
- **Quality gate**: every PDF → at least one `## Page N` heading per page; empty extracts are logged as warnings and counted under `failed`.